# 04 — Late interaction (ColBERT / MaxSim)

**Late interaction** stores *many* token vectors per chunk and scores with **MaxSim** (for each query token, take its best-matching chunk token, then sum) — finer-grained than a single pooled vector.

**Requires** `pip install -e '.[late-interaction]'` (pylate + fast-plaid + torch; Python 3.11/3.12 — already in `.venv-li`).

> **Note:** this is technically *two-stage* — a candidate set is generated, then re-ranked by MaxSim. There is no pure single-stage LI pipeline; this is the canonical way to use it. Indexing builds a fast-plaid `.plaid` sidecar, so it needs its own index.

### Index (builds the multi-vector / fast-plaid index)

In [ ]:
import sys
# Index sample_repo for this method (CLI does the heavy ingestion wiring).
# --no-inspect = read source statically (don't import the package).
!{sys.executable} -m pydocs_mcp --config configs/late_interaction.yaml index sample_repo \
    --cache-dir .pydocs-cache/late_interaction --force --no-inspect 2>&1 \
    | grep -E "Project:|Done:|rror" || true

### Search (Python pipeline)

In [ ]:
from nb_helpers import make_searcher, load_queries, show_results

# Build the retrieval PIPELINE in Python for this method (no CLI search).
search = make_searcher("configs/late_interaction.yaml", cache_dir='.pydocs-cache/late_interaction')
queries = load_queries()

In [ ]:
for q in queries:
    hits = await search(q['query'], limit=5)
    show_results(q, hits)

### Takeaway
Token-level MaxSim can out-rank single-vector dense on queries where a few precise tokens matter, at the cost of a larger index and more compute.